In [1]:
import pandas as pd 
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.multioutput import MultiOutputRegressor
from catboost import CatBoostRegressor


In [2]:
df = pd.read_csv('D:/PythonCode/DSEB65A_MachineLearningProject_Group5/data/Hanoi Daily 10 years.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
#Make copy
df2 = df.copy()

df2['day_of_year'] = df2['datetime'].dt.dayofyear
df2['month'] = df2['datetime'].dt.month
df2['day_of_week'] = df2['datetime'].dt.dayofweek
df2['is_weekend'] = (df2['day_of_week'] >= 5).astype(int)

df2['sin_day_of_year'] = np.sin(2 * np.pi * df2['day_of_year'] / 366)
df2['cos_day_of_year'] = np.cos(2 * np.pi * df2['day_of_year'] / 366)

df2['sin_month'] = np.sin(2 * np.pi * df2['month'] / 12)
df2['cos_month'] = np.cos(2 * np.pi * df2['month'] / 12)

horizons = [1, 2, 3, 4, 5]
for h in horizons:
    df2[f'target_temp_t+{h}'] = df2['temp'].shift(-h)

## Data Preprocessing

In [3]:
from sklearn.preprocessing import OneHotEncoder

#Drop insufficient collums
df2 = df2.drop(['snow', 'snowdepth', 'precipprob', 'severerisk','name', 'icon', 'description', 'sunrise', 'sunset', 'stations', 'datetime', 'solarradiation', 'uvindex', 'tempmax', 'tempmin', 'feelslikemax', 'feelslikemin', 'visibility', 'preciptype'], axis=1)
#Dealing with binary feature
# df2['preciptype'] = df2['preciptype'].fillna(value=0)
# df2['preciptype'] = df2['preciptype'].replace('rain',1)
# # One-hot encode the 'conditions' column
# # Initialize the OneHotEncoder
# encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# # Fit and transform the 'conditions' column
# encoded_conditions = encoder.fit_transform(df2[['conditions']])

# # Create a DataFrame with the encoded columns
# encoded_conditions_df = pd.DataFrame(encoded_conditions, columns=encoder.get_feature_names_out(['conditions']))

# Concatenate the encoded columns with the original DataFrame
# df2 = pd.concat([df2.reset_index(drop=True), encoded_conditions_df.reset_index(drop=True)], axis=1)
df2 = df2.drop(['conditions', 'month', 'is_weekend'], axis=1)
df2

,temp,feelslike,dew,humidity,precip,precipcover,windgust,windspeed,winddir,sealevelpressure,...,day_of_week,sin_day_of_year,cos_day_of_year,sin_month,cos_month,target_temp_t+1,target_temp_t+2,target_temp_t+3,target_temp_t+4,target_temp_t+5
0,29.3,35.4,25.7,81.5,1.400,4.17,19.4,15.7,94.8,1007.6,...,6,-0.980575,-0.196143,-1.0,-1.836970e-16,26.8,25.9,28.0,29.9,30.5
1,26.8,29.2,24.3,86.3,9.105,12.50,23.0,17.6,82.4,1006.2,...,0,-0.983798,-0.179281,-1.0,-1.836970e-16,25.9,28.0,29.9,30.5,28.6
2,25.9,26.5,24.2,90.5,31.003,12.50,20.5,18.4,56.7,1006.3,...,1,-0.986731,-0.162366,-1.0,-1.836970e-16,28.0,29.9,30.5,28.6,29.6
3,28.0,32.0,25.4,86.2,0.365,12.50,25.2,17.5,85.9,1007.0,...,2,-0.989372,-0.145404,-1.0,-1.836970e-16,29.9,30.5,28.6,29.6,30.3
4,29.9,36.4,25.9,80.4,0.100,4.17,23.4,17.5,117.7,1004.2,...,3,-0.991723,-0.128398,-1.0,-1.836970e-16,30.5,28.6,29.6,30.3,30.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3655,27.6,31.1,24.8,85.4,3.100,45.83,19.4,18.4,58.9,1007.2,...,0,-0.986731,-0.162366,-1.0,-1.836970e-16,29.3,30.0,26.0,25.7,NaN
3656,29.3,33.8,24.4,75.8,0.800,20.83,26.6,16.6,352.5,1005.5,...,1,-0.989372,-0.145404,-1.0,-1.836970e-16,30.0,26.0,25.7,NaN,NaN
3657,30.0,33.8,23.4,69.5,0.100,4.17,45.4,22.3,319.7,1002.8,...,2,-0.991723,-0.128398,-1.0,-1.836970e-16,26.0,25.7,NaN,NaN,NaN
3658,26.0,26.8,23.0,84.2,17.500,41.67,32.8,18.4,302.0,1003.2,...,3,-0.993781,-0.111355,-1.0,-1.836970e-16,25.7,NaN,NaN,NaN,NaN


In [4]:
df2.to_csv('D:/PythonCode/DSEB65A_MachineLearningProject_Group5/data/Hanoi_Daily_10_years_processed.csv', index=False)

In [5]:
print(df2.columns)
print("Number of columns:", df2.shape[1])

Index(['temp', 'feelslike', 'dew', 'humidity', 'precip', 'precipcover',
       'windgust', 'windspeed', 'winddir', 'sealevelpressure', 'cloudcover',
       'solarenergy', 'moonphase', 'day_of_year', 'day_of_week',
       'sin_day_of_year', 'cos_day_of_year', 'sin_month', 'cos_month',
       'target_temp_t+1', 'target_temp_t+2', 'target_temp_t+3',
       'target_temp_t+4', 'target_temp_t+5'],
      dtype='object')
Number of columns: 24


## Feature Engineering

In [6]:
#Tạo lag feature
for var in ['temp', 'humidity', 'dew', 'cloudcover', 'solarenergy']:
    for lag in [1, 2, 3, 4,5]:
        df2[f'{var}_lag{lag}'] = df2[var].shift(lag)

df2 = df2.dropna()
print(df2)

      temp  feelslike   dew  humidity  precip  precipcover  windgust  \
5     30.5       38.5  26.4      79.9   2.300        16.67      24.8   
6     28.6       34.3  25.6      84.2   4.225         8.33      28.1   
7     29.6       35.4  25.6      80.5   4.300        45.83      16.6   
8     30.3       36.5  25.3      76.5   0.800        20.83      20.9   
9     30.8       37.0  25.2      74.0   1.200        16.67      17.6   
...    ...        ...   ...       ...     ...          ...       ...   
3650  27.5       29.9  24.8      86.3   5.700        54.17      24.1   
3651  28.1       31.5  25.0      84.5   6.500        70.83      24.8   
3652  29.5       33.9  24.8      77.5   1.800        45.83      22.0   
3653  30.9       36.8  24.9      71.8   0.000         0.00      18.0   
3654  29.2       35.4  25.8      82.3   6.200        37.50      24.8   

      windspeed  winddir  sealevelpressure  ...  cloudcover_lag1  \
5          16.6    100.4            1003.1  ...             61.1   

## Modelling

In [7]:
# #Define X, y 
X = df2.drop([f'target_temp_t+{h}' for h in horizons], axis=1)
y = df2[[f'target_temp_t+{h}' for h in horizons]]
#Chia traint test
split_index = int(len(X) * 0.8)
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

#Build pipeline
pipeline1 = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', MultiOutputRegressor(RandomForestRegressor(random_state=42)))
])

pipeline2 = Pipeline([
    ('scaler', StandardScaler()),
    ('lgb', MultiOutputRegressor(lgb.LGBMRegressor(random_state=42)))
])

pipeline3 = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', MultiOutputRegressor(CatBoostRegressor(verbose=0, random_state=42)))
])

# Fit the pipeline
pipeline1.fit(X_train, y_train)
y_pred = pipeline1.predict(X_test)

# Calculate R-squared
r2 = r2_score(y_test, y_pred, multioutput='uniform_average')
mae = mean_absolute_error(y_test, y_pred)
print(f"✅ R² trung bình trên 5 ngày: {r2:}")
print(f"✅ MAE trung bình trên 5 ngày: {mae}°C")

# 11️⃣ In kết quả chi tiết từng ngày
for i, h in enumerate(horizons):
    r2_h = r2_score(y_test.iloc[:, i], y_pred[:, i])
    mae_h = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
    print(f"t+{h}: R²={r2_h}, MAE={mae_h}")

# 12️⃣ Dự đoán 5 ngày tới kể từ ngày cuối cùng trong dữ liệu
last_X = X.iloc[[-1]]
future_preds = pipeline1.predict(last_X)[0]

print("\n🌤️ Dự báo nhiệt độ 5 ngày tiếp theo:")
for h, val in zip(horizons, future_preds):
    print(f"Ngày t+{h}: {val}°C")

✅ R² trung bình trên 5 ngày: 0.8047385635490644
✅ MAE trung bình trên 5 ngày: 1.7268345205479452°C
t+1: R²=0.9199255927469643, MAE=1.0993
t+2: R²=0.823000275595112, MAE=1.661545205479452
t+3: R²=0.779604009795624, MAE=1.8861136986301368
t+4: R²=0.7547348194017569, MAE=1.98621095890411
t+5: R²=0.7464281202058645, MAE=2.0010027397260273

🌤️ Dự báo nhiệt độ 5 ngày tiếp theo:
Ngày t+1: 28.611000000000004°C
Ngày t+2: 28.305000000000014°C
Ngày t+3: 28.242000000000004°C
Ngày t+4: 28.221000000000004°C
Ngày t+5: 28.203000000000003°C


In [8]:
# Lấy mô hình MultiOutputRegressor từ pipeline
rf_multi = pipeline1.named_steps['rf']

# In tầm quan trọng của từng feature cho từng output
for i, estimator in enumerate(rf_multi.estimators_):
    print(f"Tầm quan trọng của feature cho output {i+1}:")
    for feature, importance in zip(X_train.columns, estimator.feature_importances_):
        print(f"{feature}: {importance:.4f}")
    print("-" * 30)

Tầm quan trọng của feature cho output 1:


temp: 0.5376
feelslike: 0.3594
dew: 0.0036
humidity: 0.0024
precip: 0.0027
precipcover: 0.0018
windgust: 0.0077
windspeed: 0.0108
winddir: 0.0062
sealevelpressure: 0.0031
cloudcover: 0.0020
solarenergy: 0.0045
moonphase: 0.0025
day_of_year: 0.0041
day_of_week: 0.0009
sin_day_of_year: 0.0020
cos_day_of_year: 0.0065
sin_month: 0.0004
cos_month: 0.0004
temp_lag1: 0.0021
temp_lag2: 0.0013
temp_lag3: 0.0021
temp_lag4: 0.0019
temp_lag5: 0.0030
humidity_lag1: 0.0015
humidity_lag2: 0.0013
humidity_lag3: 0.0016
humidity_lag4: 0.0015
humidity_lag5: 0.0015
dew_lag1: 0.0017
dew_lag2: 0.0011
dew_lag3: 0.0011
dew_lag4: 0.0015
dew_lag5: 0.0019
cloudcover_lag1: 0.0017
cloudcover_lag2: 0.0015
cloudcover_lag3: 0.0017
cloudcover_lag4: 0.0016
cloudcover_lag5: 0.0016
solarenergy_lag1: 0.0023
solarenergy_lag2: 0.0015
solarenergy_lag3: 0.0015
solarenergy_lag4: 0.0015
solarenergy_lag5: 0.0016
------------------------------
Tầm quan trọng của feature cho output 2:
temp: 0.3281
feelslike: 0.3772
dew: 0.0072
hu